In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# 파일 경로 설정
file_a_path = "Your DATA Aset.csv"
file_b_path = "Your DATA Bset.csv"

In [ ]:
df_a = pd.read_csv(file_a_path)
df_b = pd.read_csv(file_b_path)

# 질문 컬럼 가정
col_a = 'QuestionA'
col_b = 'QuestionB'

list_A = df_a[col_a].astype(str).tolist()
list_B = df_b[col_b].astype(str).tolist()

print(f"List A length: {len(list_A)}")
print(f"List B length: {len(list_B)}")

In [ ]:


model = SentenceTransformer("intfloat/multilingual-e5-large-instruct5")   #임베딩 모델 변경(Eebedings model change)
emb_A = model.encode(list_A, show_progress_bar=True, batch_size=64)
emb_B = model.encode(list_B, show_progress_bar=True, batch_size=64)


In [ ]:
import pandas as pd
from sentence_transformers import util
from pathlib import Path

# list_A, list_B : 질문 텍스트 리스트
# emb_A, emb_B   : 같은 순서의 임베딩 텐서 (질문당 1개)

# 1) 코사인 유사도 행렬
cosine_scores = util.cos_sim(emb_A, emb_B)  # shape: [len(A), len(B)]

# 2) DataFrame으로 변환 (인덱스/컬럼 부여)
index_A = [f"A_{i}" for i in range(len(list_A))]
index_B = [f"B_{j}" for j in range(len(list_B))]
sim_df = pd.DataFrame(cosine_scores.cpu().numpy(), index=index_A, columns=index_B)
# 4) CSV 저장
Path("analysis_results").mkdir(exist_ok=True)
sim_df.to_csv(f"cosine_similarity_matrix_{model}.csv", encoding="utf-8-sig")

print(f"saved: cosine_similarity_matrix_{model}.csv")

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from kneed import KneeLocator

out_dir = Path("analysis_results"); out_dir.mkdir(exist_ok=True)

# 1) 행렬 로드 + 음수→0 클립
mat = pd.read_csv(f"analysis_results/cosine_similarity_matrix_{model}.csv", index_col=0).clip(lower=0)

# 2) 문장쌍 분포
vals = mat.values.ravel()
print(f"pairs={vals.size}, min={vals.min():.4f}, max={vals.max():.4f}")
print("quantiles:\n", pd.Series(vals).quantile([0,0.5,0.9,0.95,0.99,1]))

# 3) A별 최고/평균
best_per_A = mat.max(axis=1)
mean_per_A = mat.mean(axis=1)

# 4) best 기준 정렬 
order = best_per_A.sort_values(ascending=False).index
target_series = best_per_A.loc[order].reset_index(drop=True)  # 이전 mean_sorted 대신 사용


# 5) 질문 텍스트 로드
aset = pd.read_csv("Data/Aset.csv")
bset = pd.read_csv("Data/Bset.csv")
map_A = aset["QuestionA"].to_dict()
map_B = bset["QuestionB"].to_dict() if "QuestionB" in bset.columns else bset.iloc[:,0].to_dict()

# 6) A별 요약 (best에 대응하는 B 포함)
rows = []
for a_idx in mat.index:
    sims = mat.loc[a_idx]
    best_b = sims.idxmax()
    best_sim = sims[best_b]
    mean_val = sims.mean()
    mean_b = (sims - mean_val).abs().idxmin()  # 평균에 가장 가까운 B
    rows.append({
        "index_A": a_idx,
        "question_A": map_A.get(int(a_idx.split("_")[1]), ""),
        "best_index_B": best_b,
        "best_question_B": map_B.get(int(best_b.split("_")[1]), ""),
        "best_similarity": best_sim,
        "mean_similarity": mean_val,
        "mean_index_B": mean_b,
        "mean_question_B": map_B.get(int(mean_b.split("_")[1]), ""),
    })

summary = pd.DataFrame(rows)
summary.to_csv(out_dir/f"{model}_a_best_mean_clipped.csv", index=False, encoding="utf-8-sig")


# 7) 클립된 행렬 저장
mat.to_csv(out_dir/f"{model}_best_similarity_matrix_clipped.csv", encoding="utf-8-sig")

print("saved:",
      out_dir/f"{model}_best_similarity_matrix_clipped.csv",
      out_dir/f"{model}_best_a_best_mean_clipped.csv",
      out_dir/f"{model}_best_a_below_best_knee_clipped.csv",
      out_dir/f"{model}_best_per_A_knees.png")

In [ ]:
#유사도행렬을 기준으로 임계값 확인 90% 분위수 계산 및 저장  #by model
import pandas as pd

mat = pd.read_csv("Projects/ADHD2/analysis_results/{model}/best/{model}_best_similarity_matrix_clipped.csv", index_col=0)
vals = mat.values.ravel()  # A×B 모든 유사도
threshold = pd.Series(vals).quantile(0.90) #곡선 분위수 90%  etc.

print(f"count={vals.size}")
print(f"90% quantile similarity threshold: {threshold:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
#Figure1
# =========================
# 1) 파일 경로
# =========================
paths = {
    "MBERT": r"Projects/ADHD2/analysis_results/{model}/best/{model}_best_similarity_matrix_clipped.csv",
    "LaBSE": r"Projects/ADHD2/analysis_results/{model}/best/{model}_best_similarity_matrix_clipped.csv",
    "mE5L":  r"Projects/ADHD2/analysis_results/{model}/best/{model}_best_similarity_matrix_clipped.csv",
}

OUT_PLOT = r"Projects/ADHD2/valid/similarity_percent_hist_fancy.png"
OUT_TABLE = r"Projects/ADHD2/valid/similarity_bin_table_grouped.csv"

# Q90 threshold
q90_reported = {"MBERT": 0.7522, "LaBSE": 0.5912, "mE5L": 0.8879}

# =========================
# 2) 설정
# =========================
bins = 80
group_bins = np.arange(0.0, 1.0 + 0.1, 0.1)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Times"],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "figure.dpi": 300
})

colors = {
    "MBERT": "#4C78A8",
    "LaBSE": "#B07AA1",
    "mE5L":  "#E3B98F",
}

# =========================
# 3) 데이터 로드
# =========================
scores = {}
for name, p in paths.items():
    mat = pd.read_csv(p, index_col=0)
    vals = mat.to_numpy().ravel()
    scores[name] = vals

all_vals = np.concatenate(list(scores.values()))
bin_edges = np.linspace(all_vals.min(), all_vals.max(), bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]

# =========================
# 4) Percent 히스토그램
# =========================
fig, ax = plt.subplots(figsize=(8.2, 5.2))

# 여백 확보 (오른쪽 라벨 공간)
ax.set_xlim(all_vals.min(), all_vals.max() + 0.03)

for name in ["MBERT", "LaBSE", "mE5L"]:
    vals = scores[name]
    counts, _ = np.histogram(vals, bins=bin_edges)
    percents = counts / counts.sum() * 100

    ax.bar(
        bin_centers, percents, width=bin_width,
        alpha=0.55, label=name,
        color=colors[name], edgecolor="white", linewidth=0.3
    )

    # Q90 점선 + 라벨 (점선 옆)
for name in ["MBERT", "LaBSE", "mE5L"]:
    q90 = q90_reported[name]
    ax.axvline(q90, linestyle="--", linewidth=1.0, color=colors[name])

    


# 축/그리드
ax.set_xlabel("Similarity score")
ax.set_ylabel("Percent (%)")
##ax.set_title("Similarity score distribution (Percent histogram)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.25)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Q90 라벨: 검은색, 오른쪽 여백에 위치
ymax = ax.get_ylim()[1]
label_x = all_vals.max() + 0.015
label_y_positions = [0.34, 0.3, 0.26]  # 겹침 방지

for i, name in enumerate(["MBERT", "LaBSE", "mE5L"]):
    q90 = q90_reported[name]
    ax.text(
        label_x, ymax * label_y_positions[i],
        f"{name} 90% threshold = {q90:.4f}",
        ha="left", va="top", fontsize=8, color="black"
    )

ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig(OUT_PLOT, dpi=300)
plt.show()

# =========================
# 5) 구간 합친 표 (Count + Percent)
# =========================
rows = []
for name in ["MBERT", "LaBSE", "mE5L"]:
    vals = scores[name]
    counts, _ = np.histogram(vals, bins=group_bins)
    percents = counts / counts.sum() * 100

    for i in range(len(group_bins) - 1):
        rows.append({
            "Model": name,
            "Range": f"{group_bins[i]:.1f}-{group_bins[i+1]:.1f}",
            "Count": int(counts[i]),
            "Percent": round(percents[i], 2)
        })

table_df = pd.DataFrame(rows)

pivot_count = table_df.pivot(index="Range", columns="Model", values="Count")
pivot_percent = table_df.pivot(index="Range", columns="Model", values="Percent")

final_rows = []
for rng in pivot_count.index:
    row = {"Range": rng}
    for model in pivot_count.columns:
        c = pivot_count.loc[rng, model]
        p = pivot_percent.loc[rng, model]
        row[model] = f"{c} ({p:.2f}%)"
    final_rows.append(row)

final_df = pd.DataFrame(final_rows)
final_df.to_csv(OUT_TABLE, index=False, encoding="utf-8-sig")

print("Saved plot:", OUT_PLOT)
print("Saved table:", OUT_TABLE)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from kneed import KneeLocator
from datetime import datetime
#Figure2

out_dir = Path(r"Projects\ADHD2\analysis_results")
out_dir.mkdir(exist_ok=True)

date_str = datetime.now().strftime("%Y%m%d")

model_configs = {
    "mE5L": {
        "full_name": "intfloat/multilingual-e5-large-instruct",
        "matrix_path": r"Projects\ADHD2\analysis_results\mE5L\best\mE5L_best_similarity_matrix_clipped.csv",
        "color": "royalblue",
    },
    "MBERT": {
        "full_name": "bert-base-multilingual-cased",
        "matrix_path": r"Projects\ADHD2\analysis_results\MBERT\best\MBERT_best_similarity_matrix_clipped.csv",
        "color": "darkorange",
    },
    "LaBSE": {
        "full_name": "sentence-transformers/LaBSE",
        "matrix_path": r"Projects\ADHD2\analysis_results\LaBSE\best\LaBSE_best_similarity_matrix_clipped.csv",
        "color": "seagreen",
    },
}

S = 1.0

all_best_match_dfs = []
kneedle_rows = []

# =========================
# 1) 모델별 best-match curve 계산
# =========================

for model_name, config in model_configs.items():
    mat = pd.read_csv(config["matrix_path"], index_col=0).clip(lower=0)

    print("=" * 70)
    print(f"Model: {model_name}")
    print(f"Matrix shape: {mat.shape}")

    best_matches = mat.max(axis=0)
    best_match_faqs = mat.idxmax(axis=0)

    sorted_similarities = best_matches.sort_values(ascending=False)

    best_match_df = pd.DataFrame({
        "model": model_name,
        "PublicQ": sorted_similarities.index,
        "Best_Match_Similarity": sorted_similarities.values,
        "Matching_FAQ": best_match_faqs[sorted_similarities.index].values,
    }).reset_index(drop=True)

    best_match_df["Rank"] = best_match_df.index + 1

    x = best_match_df["Rank"].values
    y = best_match_df["Best_Match_Similarity"].values

    kl = KneeLocator(
        x,
        y,
        curve="concave",
        direction="decreasing",
        S=S,
    )

    if kl.knee is not None:
        knee_rank = int(kl.knee)
        kneedle_threshold = float(kl.knee_y)
    else:
        knee_rank = None
        kneedle_threshold = None

    kneedle_rows.append({
        "model": model_name,
        "S": S,
        "knee_rank": knee_rank,
        "kneedle_threshold": kneedle_threshold,
    })

    all_best_match_dfs.append(best_match_df)

all_best_match_df = pd.concat(all_best_match_dfs, ignore_index=True)
kneedle_df = pd.DataFrame(kneedle_rows)

print("\n=== Kneedle results ===")
print(kneedle_df.to_string(index=False))

# =========================
# 2) 한 figure에 3개 모델 curve 그리기
# =========================

plt.figure(figsize=(13, 8))

for model_name, config in model_configs.items():
    model_df = all_best_match_df[
        all_best_match_df["model"] == model_name
    ].copy()

    x = model_df["Rank"].values
    y = model_df["Best_Match_Similarity"].values

    plt.plot(
        x,
        y,
        label=f"{model_name} best-match curve",
        color=config["color"],
        linewidth=2,
        alpha=0.9,
    )

    row = kneedle_df[kneedle_df["model"] == model_name].iloc[0]

    if pd.notna(row["knee_rank"]) and pd.notna(row["kneedle_threshold"]):
        knee_rank = int(row["knee_rank"])
        threshold = float(row["kneedle_threshold"])

        plt.scatter(
            [knee_rank],
            [threshold],
            s=90,
            color=config["color"],
            edgecolor="black",
            zorder=5,
            label=f"{model_name} knee: rank={knee_rank}, t={threshold:.4f}",
        )

        plt.vlines(
            knee_rank,
            ymin=0,
            ymax=threshold,
            colors=config["color"],
            linestyles="dashed",
            alpha=0.45,
        )

        plt.hlines(
            threshold,
            xmin=0,
            xmax=knee_rank,
            colors=config["color"],
            linestyles="dotted",
            alpha=0.45,
        )

plt.title(
    "Best-Match Similarity Curves and Kneedle Thresholds by Model",
    fontsize=16,
)

plt.xlabel(
    "Public Question Rank (sorted by best-match similarity within each model)",
    fontsize=12,
)

plt.ylabel(
    "Best-Match Similarity Score",
    fontsize=12,
)

plt.xlim(left=0)
plt.ylim(0, 1.05)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(fontsize=9)
plt.tight_layout()

fig_path = out_dir / f"best_match_similarity_curves_all_models_kneedle_{date_str}.png"

plt.savefig(fig_path, dpi=300)
plt.show()

# =========================
# 3) 결과 저장
# =========================

best_match_path = out_dir / f"best_match_details_all_models_{date_str}.csv"
kneedle_path = out_dir / f"kneedle_thresholds_all_models_{date_str}.csv"

all_best_match_df.to_csv(
    best_match_path,
    index=False,
    encoding="utf-8-sig",
)

kneedle_df.to_csv(
    kneedle_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved files:")
print(f" - Figure: {fig_path}")
print(f" - Best match details: {best_match_path}")
print(f" - Kneedle thresholds: {kneedle_path}")